# Notebook 09 — Async IO for AI services

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Advanced |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `08-advanced-typing-protocols-dataclasses.ipynb`

**Next up:** `10-testing-debugging.ipynb`

---

## Learning objectives

- Overlap I/O-bound waits with `asyncio`.
- Use `gather` for parallel fetches.
- Know blocking vs async boundaries before FastAPI.

## Table of contents

1. **Coroutines recap**
2. **`gather` parallelism**
3. **`asyncio.to_thread` offload**
4. **Progressive drills — seq vs parallel → timeouts → tolerant gather**
5. **Exercise — first successful coroutine**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


## 1 · Concurrent sleeps

*Explanation:* Replace sleeps with HTTP clients later — structure identical.


In [ ]:
import asyncio

async def fetch(name: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return f"{name}-done"

async def main():
    out = await asyncio.gather(
        fetch("a", 0.15),
        fetch("b", 0.15),
        fetch("c", 0.15),
    )
    print(out)

await main()


> If `await main()` fails in your runtime, run `asyncio.run(main())` instead.


## 2 · Semaphore limiting

*Explanation:* Bound concurrency — critical when calling rate-limited APIs.


In [ ]:
import asyncio

async def bounded_gather(items: list[int], limit: int):
    sem = asyncio.Semaphore(limit)

    async def work(i: int) -> int:
        async with sem:
            await asyncio.sleep(0.05)
            return i * i

    return await asyncio.gather(*(work(i) for i in items))

print(await bounded_gather(list(range(6)), limit=2))


## 3 · `asyncio.to_thread` offload

*Explanation:* Use a thread pool for **blocking** CPU or C-extension work so the event loop can still schedule other coroutines — common when bridging `requests`, file parsing, or numpy inside async services.


In [ ]:
import asyncio

def blocking_sum(n: int) -> int:
    return sum(range(n))

async def run_offload():
    return await asyncio.to_thread(blocking_sum, 200_000)

print(await run_offload())


---

## Progressive drills — **easy → harder**

Async separates **latency-bound** waits—practice measuring overlap, enforcing deadlines, and isolating flaky vendors.


### A · Easiest — sequential vs overlapping sleeps

Wall-clock drops when I/O overlaps—this is why FastAPI loves `async`.


In [ ]:
import asyncio
import time


async def sequential_sleeps() -> float:
    t0 = time.perf_counter()
    await asyncio.sleep(0.06)
    await asyncio.sleep(0.06)
    return time.perf_counter() - t0


async def parallel_sleeps() -> float:
    t0 = time.perf_counter()
    await asyncio.gather(asyncio.sleep(0.06), asyncio.sleep(0.06))
    return time.perf_counter() - t0


async def bench_sleep() -> None:
    print("sequential ~", round(await sequential_sleeps(), 3), "s")
    print("parallel   ~", round(await parallel_sleeps(), 3), "s")


await bench_sleep()


### B · Medium — hard deadlines with `wait_for`

Stop runaway vendor calls politely.


In [ ]:
import asyncio


async def slow_vendor() -> str:
    await asyncio.sleep(1.0)
    return "never"


try:
    await asyncio.wait_for(slow_vendor(), timeout=0.07)
except asyncio.TimeoutError:
    print("cancelled slow vendor")


### C · Harder — `gather(..., return_exceptions=True)`

One poisoned task shouldn't nuke an entire fan-out batch.


In [ ]:
import asyncio


async def flaky_shard() -> str:
    raise ValueError("timeout from AZ-1")

async def healthy_shard() -> str:
    return "chunk-42"


mixed = await asyncio.gather(flaky_shard(), healthy_shard(), return_exceptions=True)
print(mixed)


### Exercise — first successful

`async def first_done(coro_a, coro_b)` schedules both coroutines and returns the **result** of whichever finishes first. Use `asyncio.create_task`, `asyncio.wait(..., return_when=asyncio.FIRST_COMPLETED)`, cancel the slower one, then `await` the winner.


In [ ]:
import asyncio

async def slow_return(val: int, delay: float) -> int:
    await asyncio.sleep(delay)
    return val

async def first_done(a, b):
    raise NotImplementedError


print(await first_done(slow_return(1, 0.05), slow_return(2, 0.2)))


### Solution — first_done

<details>
<summary>Click to expand</summary>

```python
async def first_done(a, b):
    t1 = asyncio.create_task(a)
    t2 = asyncio.create_task(b)
    done, pending = await asyncio.wait({t1, t2}, return_when=asyncio.FIRST_COMPLETED)
    winner = done.pop()
    for p in pending:
        p.cancel()
    return await winner
```

</details>
